[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C55_TSR_Autonomous_Driving_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（针孔模型 / 像素预算 / 相机选型反解 / 代价不对称）

目标：在写任何模型代码之前，先用**物理量**把 TSR 的边界算清楚——
**多远的标志有多少像素、安全需要多远检出、什么相机配置才够、哪种错误更贵**。

本 notebook 你会亲手实现：
1. **针孔相机模型**：从 (图像宽, HFOV) 求等效焦距（像素）
2. **像素-距离换算器**：标志在任意距离下的成像尺寸与面积占比
3. **小目标判定**：算出「从多远开始，这块牌就是 COCO 定义的 small」
4. **安全反推**：从「100→60 km/h 要多少米」反推所需检出距离与像素
5. **五个本质不同的量化速览**（小·偏·硬·连·歪）
6. ✏️ 相机选型反解器 / 接近过程可用帧数 / 代价敏感风险打分

> 心智模型：**安全需要 146 米，光学只给 7 像素。这门课的六个模块都是在补这个缺口。**

## 1 · 环境自检

In [ ]:
import sys, platform, math, json, itertools, collections
import numpy as np

print('Python', sys.version.split()[0], '|', platform.system(), platform.machine())
print('numpy ', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
_ver = tuple(int(x) for x in np.__version__.split('.')[:2])
assert _ver >= (1, 20), 'numpy 版本过低'

rng = np.random.default_rng(55)
print('\n✅ 环境就绪：本课全程 **纯 numpy + 标准库、CPU、不联网**。')
print('   我们不合成图像，只合成**物理量**——相机参数、标志尺寸、车速都用真实值。')

## 2 · 针孔相机模型：从视场角求等效焦距

$$f_{px}=\frac{W/2}{\tan(\mathrm{HFOV}/2)}$$

**这个量是 TSR 所有定量推理的起点**：它把「相机选型」和「感知能力」连成一条公式。

In [ ]:
def focal_px(img_w, hfov_deg):
    '''等效焦距（像素单位）。img_w: 图像宽（px）；hfov_deg: 水平视场角（度）。'''
    return (img_w / 2.0) / math.tan(math.radians(hfov_deg) / 2.0)

# 量产车上常见的四种前视配置
CAMERAS = {
    '广角 120° · 1920x1080': (1920, 1080, 120.0),
    '主摄  60° · 1920x1080': (1920, 1080,  60.0),
    '主摄  60° · 3840x2160': (3840, 2160,  60.0),
    '长焦  30° · 1920x1080': (1920, 1080,  30.0),
}

print(f"{'相机配置':<26s} {'f_px':>9s}   说明")
for name, (w, h, fov) in CAMERAS.items():
    print(f'{name:<26s} {focal_px(w, fov):>9.1f}')

# 数值校核
assert abs(focal_px(1920, 60.0) - 1662.77) < 0.5
assert abs(focal_px(1920, 30.0) - 3582.35) < 0.5
assert np.isclose(focal_px(3840, 60.0), 2 * focal_px(1920, 60.0))   # 分辨率翻倍 → 焦距翻倍
assert focal_px(1920, 120.0) < focal_px(1920, 60.0) < focal_px(1920, 30.0)
print('\n✅ 视场角越窄 → 焦距越长 → 同一目标成像越大（代价是看得越窄）')
print('✅ 同视场角下分辨率翻倍 ⇔ 焦距翻倍 ⇔ **有效识别距离翻倍**，代价是算力约 4 倍')

## 3 · 像素-距离换算器：标志在图像里到底有多大

$$p=\frac{f_{px}\cdot S}{Z}$$

标志的物理尺寸 S 来自 **GB 5768**：城区圆形禁令牌直径 60 cm，公路 80 cm，
三角警告牌边长 70 cm（郊区 90 cm）。

In [ ]:
def sign_px(f_px, size_m, dist_m):
    '''标志在图像中的成像边长（像素）。'''
    return f_px * size_m / dist_m

SIGN_SIZES = {          # GB 5768 常见规格（米）
    '圆形禁令牌 Ø60cm(城区)': 0.60,
    '圆形禁令牌 Ø80cm(公路)': 0.80,
    '三角警告牌 边长70cm':    0.70,
}

S = 0.60                                     # 本 notebook 的基准标志
f_main = focal_px(1920, 60.0)                # 基准相机：主摄 60° 1080p
DISTS = [20, 30, 50, 60, 80, 100, 146]

print(f'基准：{S*100:.0f} cm 圆形标志 · 主摄 60° 1920x1080 (f={f_main:.0f} px)\n')
print(f"{'距离(m)':>8s} {'边长(px)':>10s} {'面积(px^2)':>12s} {'占全图':>9s}")
for z in DISTS:
    p = sign_px(f_main, S, z)
    frac = 100.0 * p * p / (1920 * 1080)
    print(f'{z:>8d} {p:>10.1f} {p*p:>12.0f} {frac:>8.4f}%')

assert abs(sign_px(f_main, S, 60) - 16.63) < 0.05
assert abs(sign_px(f_main, S, 100) -  9.98) < 0.05
assert abs(sign_px(f_main, S, 146) -  6.83) < 0.05
# 反比关系：距离翻倍 → 像素减半 → 面积变 1/4
assert np.isclose(sign_px(f_main, S, 50) / sign_px(f_main, S, 100), 2.0)
print('\n⚠️  60 米外只有 **17 像素**，占全图面积 **0.013%**（约 1/7500）。')
print('    「16–30 px 是 TSR 的常态而非极端」——这句话是整门课的出发点。')

## 4 · 小目标判定：从多远开始它就是 COCO 定义的 small

COCO 的定义：面积 < 32² = 1024 px² 为 small，32²–96² 为 medium，> 96² 为 large。
**关键问题不是「它是不是小目标」，而是「从多远开始它变成小目标」。**

In [ ]:
def coco_bucket(px):
    a = px * px
    return 'small' if a < 32**2 else ('medium' if a < 96**2 else 'large')

def dist_for_px(f_px, size_m, target_px):
    '''反解：标志成像达到 target_px 时对应的距离。'''
    return f_px * size_m / target_px

print(f"{'相机配置':<26s} {'变 small 的距离':>16s} {'降到 16px 的距离':>18s} {'降到 8px':>12s}")
for name, (w, h, fov) in CAMERAS.items():
    f = focal_px(w, fov)
    print(f'{name:<26s} {dist_for_px(f, S, 32):>14.1f} m {dist_for_px(f, S, 16):>16.1f} m'
          f' {dist_for_px(f, S, 8):>10.1f} m')

z32 = dist_for_px(f_main, S, 32)
assert abs(z32 - 31.18) < 0.05                # 主摄 60°：超过 31 米就是 small
assert coco_bucket(sign_px(f_main, S, 20)) == 'medium'
assert coco_bucket(sign_px(f_main, S, 40)) == 'small'
print(f'\n⚠️  主摄 60° 下，**超过 {z32:.0f} 米这块牌就落进 COCO 的 small 桶**。')
print('    而 TSR 真正需要的工作距离是 60–150 米 —— 也就是说：')
print('    **TSR 的绝大部分工作量都发生在「比 COCO small 还小」的区域。**')
print('    这直接意味着：按 COCO 调好的 anchor/stride/IoU 阈值，拿到 TSR 上必然要重调。')

## 5 · 从安全需求反推检出距离：本课的中心矛盾

$$D = v_0 t_r + \frac{v_0^2-v_1^2}{2a}$$

$t_r$ 是**总反应时间**（感知确认 + 决策 + 执行器响应），$a$ 是可接受的减速度。
乘用车舒适减速度约 2 m/s²，紧急制动可到 6–8 m/s² 但会明显影响体验。

In [ ]:
def required_distance(v0_kmh, v1_kmh, decel=2.0, reaction_s=0.8):
    '''从 v0 减到 v1 所需的纵向距离（米）。'''
    v0, v1 = v0_kmh / 3.6, v1_kmh / 3.6
    return v0 * reaction_s + max(0.0, (v0**2 - v1**2) / (2 * decel))

SCENARIOS = [
    ('高速 120 -> 100 (限速降级)', 120, 100, 2.0, 0.8),
    ('高速 100 ->  60 (施工区)  ', 100,  60, 2.0, 0.8),
    ('高速 100 ->  60 (可接受急一点)', 100, 60, 3.0, 0.8),
    ('城区  60 ->  30 (学校区域)', 60,  30, 2.0, 0.8),
    ('城区  50 ->   0 (停车让行)', 50,   0, 3.0, 0.8),
]

print(f"{'场景':<32s} {'所需距离':>10s} {'主摄60°像素':>13s} {'长焦30°像素':>13s}")
for name, v0, v1, a, tr in SCENARIOS:
    D = required_distance(v0, v1, a, tr)
    p_main = sign_px(focal_px(1920, 60.0), S, D)
    p_tele = sign_px(focal_px(1920, 30.0), S, D)
    print(f'{name:<32s} {D:>9.1f} m {p_main:>12.1f} {p_tele:>12.1f}')

D_key = required_distance(100, 60, 2.0, 0.8)
assert abs(D_key - 145.68) < 0.1
assert abs(sign_px(focal_px(1920, 60.0), S, D_key) - 6.85) < 0.05
assert sign_px(focal_px(1920, 30.0), S, D_key) > 2 * sign_px(focal_px(1920, 60.0), S, D_key)
print(f'\n⚠️  **本课的中心矛盾**：100->60 km/h 需要 {D_key:.0f} 米的提前量，')
print(f'    而那里主摄只给 {sign_px(focal_px(1920,60.0), S, D_key):.1f} 像素 —— '
      '「60」和「80」的区别在物理上不存在。')
print('✅ 出路只有三条，且必须组合使用：')
print('   ① 换硬件：长焦相机 / 更高分辨率（有效距离随焦距线性增长）')
print('   ② 用时序：远处先输出 L1 粗类，随着接近逐帧累积到 L2 细类（模块 04）')
print('   ③ 用先验：高精地图的标志图层给出「这里应该有一块牌」（模块 04）')
print('   —— **没有一条是「换更大的 backbone」**。这就是 TSR 领域知识的价值。')

## 6 · 五个本质不同的量化速览：小 · 偏 · 硬 · 连 · 歪

上面已经量化了「小」。这一节把另外四个也各用几行代码量化一次，
**目的是让这个框架从「五句话」变成「五个数」**——面试里说得出数字才可信。

In [ ]:
# ── ① 小：IoU 对位移的敏感性（同样偏移 2 px，小框被判死、大框没事）
def iou_shift(side, delta):
    '''边长 side 的正方形框在 x,y 各偏移 delta 后与原框的 IoU。'''
    inter = max(0.0, side - delta) ** 2
    return inter / (2 * side**2 - inter)

print('① 小 —— 同样偏移 2 px：')
for s_ in [8, 16, 32, 64]:
    print(f'   {s_:>3d}x{s_:<3d} 框  IoU = {iou_shift(s_, 2):.3f}'
          f'   {"❌ 低于 0.5 阈值，直接被判为负样本" if iou_shift(s_, 2) < 0.5 else ""}')
assert abs(iou_shift(8, 2) - 36/92) < 1e-9
assert abs(iou_shift(64, 2) - 0.8842) < 1e-3
print('   → IoU=0.5 对 8px 框要求「亚 2 像素」定位，对 64px 框只要求「10 像素以内」。')
print('   → **同一个阈值，两个数量级的严苛度差异** —— 小目标正样本稀缺的根因。\n')

# ── ② 偏：长尾（Zipf），且尾巴随地域改变
ranks = np.arange(1, 121)
w = ranks ** -1.6; w = w / w.sum()
print('② 偏 —— 120 个细类、Zipf(s=1.6) 的类别分布：')
print(f'   前 10 类占全部实例的 {100*w[:10].sum():.1f}%')
print(f'   后 60 类合计只占     {100*w[60:].sum():.1f}%')
print(f'   头尾频次比            {w[0]/w[-1]:.0f}x')
assert w[:10].sum() > 0.75 and w[60:].sum() < 0.05
print('   → 而且这条尾巴是**区域相关**的：「注意牲畜」在西部国道是常见类。\n')

# ── ③ 硬：错分直接变成错误的控制动作
CONFUSABLE = [('限速60', '限速80', '车按 80 开，超速'),
              ('限速80', '限速60', '无故减速，用户投诉'),
              ('禁止驶入', '禁止停车', '错过路口/绕路'),
              ('停车让行', '减速让行', '路口不停车，安全事件')]
print('③ 硬 —— 错分不是「掉分」，是错误的控制动作：')
for a, b, effect in CONFUSABLE:
    print(f'   {a} → {b:<8s}  下游后果：{effect}')
print('   → **混淆矩阵比 mAP 重要**：你必须知道哪几对类互相错分。\n')

# ── ④ 连：接近过程中有多少帧可用
def approach_seconds(z_far, z_near, speed_kmh):
    return (z_far - z_near) / (speed_kmh / 3.6)
t_ = approach_seconds(100, 30, 100.0)
print('④ 连 —— 标志静止 + 自车运动已知：')
print(f'   100 km/h 下从 100 m 接近到 30 m 用时 {t_:.2f} s，30 FPS 下有 '
      f'{math.floor(t_*30 + 1e-9)} 帧观测')
assert math.floor(t_ * 30 + 1e-9) == 75
print('   → 而行人检测拿不到这个红利（目标自己在动，未来位置不可解析预测）。')
print('   → **TSR 的跟踪本质上是自车运动补偿，而不是目标运动建模。**\n')

# ── ⑤ 歪：双重不对称（FN vs FP，且按类别再次不对称）
COST = {'停车让行': {'fn': 1000, 'fp':  50},
        '限速60':   {'fn':  100, 'fp':  30},
        '景点指示': {'fn':    1, 'fp':   1}}
print('⑤ 歪 —— 代价的双重不对称（相对单位）：')
print(f"   {'类别':<10s} {'漏检代价':>9s} {'误检代价':>9s} {'FN/FP 比':>10s}")
for k, v in COST.items():
    print(f"   {k:<10s} {v['fn']:>9d} {v['fp']:>9d} {v['fn']/v['fp']:>10.1f}x")
assert COST['停车让行']['fn'] / COST['景点指示']['fn'] == 1000
print('   → 关键类与非关键类的漏检代价相差 1000 倍，而 mAP 对它们**一视同仁**。')
print('   → 这就是模块 05「安全导向评测」存在的理由。')

## ✏️ 练习 1：相机选型反解器

实现 `required_hfov_deg(size_m, dist_m, min_px, img_w)`：
给定「物理尺寸 `size_m` 的标志，必须在 `dist_m` 米外达到 `min_px` 像素」，
求所需的**最大**水平视场角（度）。

推导：`f = min_px * dist_m / size_m`，`HFOV = 2 * arctan((img_w/2) / f)`。

这是把「感知需求」翻译成「相机选型」的那一步——面试里能当场推出来非常加分。

In [ ]:
def required_hfov_deg(size_m, dist_m, min_px, img_w=1920):
    # TODO: ① 求所需焦距 f（像素）  ② 由 f 与 img_w 反解 HFOV（度）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
h1 = required_hfov_deg(0.60, 100.0, 20, 1920)
h2 = required_hfov_deg(0.60,  50.0, 20, 1920)
h3 = required_hfov_deg(0.60, 100.0, 30, 1920)
assert abs(h1 - 32.15) < 0.2, f'60cm 标志 @100m 达 20px 需要约 32° 视场，得到 {h1}'
assert abs(h2 - 59.87) < 0.2, f'@50m 达 20px 恰好对应约 60° 主摄，得到 {h2}'
assert h3 < h1, '要求更多像素 → 视场必须更窄'
assert required_hfov_deg(0.60, 100.0, 20, 3840) > h1, '分辨率翻倍 → 同需求下可用更宽视场'
print(f"{'需求':<34s} {'所需 HFOV':>10s}")
for s_, z_, p_, w_ in [(0.6, 50, 20, 1920), (0.6, 100, 20, 1920),
                       (0.6, 146, 20, 1920), (0.6, 100, 20, 3840)]:
    print(f'{f"{s_*100:.0f}cm @ {z_}m >= {p_}px, W={w_}":<34s} '
          f'{required_hfov_deg(s_, z_, p_, w_):>9.1f}°')
print('\n✅ 练习 1 通过：**「要在 100 m 外识别限速牌」= 「要一路 32° 的长焦相机」**')
print('   —— 量产车上那颗 30° 长焦的来历，就是这一行公式。')

## ✏️ 练习 2：接近过程的可用帧数

实现 `approach_frames(z_far, z_near, speed_kmh, fps)`：
车以 `speed_kmh` 匀速接近，标志从 `z_far` 米到 `z_near` 米之间共有多少帧观测。

用 `t = (z_far - z_near) / (speed_kmh / 3.6)`，
帧数 `= math.floor(t * fps + 1e-9)`（+1e-9 是为了避开浮点误差，必须照写）。

这个数决定了模块 04「多帧融合」能有多少本钱。

In [ ]:
def approach_frames(z_far, z_near, speed_kmh, fps=30.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert approach_frames(100, 30, 100.0, 30.0) == 75
assert approach_frames(100, 30, 120.0, 30.0) == 63
assert approach_frames(100, 30,  60.0, 30.0) == 126
assert approach_frames(100, 30, 100.0, 15.0) == 37      # 帧率减半 → 帧数减半
assert approach_frames( 60, 30, 100.0, 30.0) == 32      # 只从 60m 起算 → 帧数骤减
print(f"{'场景':<34s} {'可用帧数':>9s}")
for zf, zn, v, f_ in [(100, 30, 100, 30), (100, 30, 120, 30), (100, 30, 60, 30),
                      (60, 30, 100, 30), (100, 30, 100, 15)]:
    print(f'{f"{zf}->{zn} m @ {v} km/h, {f_} FPS":<34s} '
          f'{approach_frames(zf, zn, v, f_):>9d}')
print('\n✅ 练习 2 通过。三条可迁移的结论：')
print('   ① 车速越高，可用帧数越少 —— **高速工况才是 TSR 最难的工况**（信息更少、要求更远）')
print('   ② 帧率是可用信息量的线性因子，但提高帧率也线性提高算力')
print('   ③ **能多早开始检出，比单帧精度更值钱**：首检距离从 60m 推到 100m，')
print('      可用帧数从 32 涨到 75 —— 多帧融合的空间直接翻倍。')

## ✏️ 练习 3：代价敏感的风险打分

实现 `risk_score(errors, costs)`：

- `errors = {类别: {'fn': 漏检次数, 'fp': 误检次数}}`
- `costs  = {类别: {'fn': 单次漏检代价, 'fp': 单次误检代价}}`
- 返回 `{类别: fn*cost_fn + fp*cost_fp}`

**这是模块 05 的雏形**：它回答的是「下一个季度先修哪个类」，
而 mAP 回答不了这个问题。

In [ ]:
def risk_score(errors, costs):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ERRORS = {'停车让行': {'fn':   2, 'fp':   1},
          '限速60':   {'fn':  30, 'fp':  20},
          '景点指示': {'fn': 200, 'fp': 150}}
COSTS  = {'停车让行': {'fn': 1000, 'fp': 50},
          '限速60':   {'fn':  100, 'fp': 30},
          '景点指示': {'fn':    1, 'fp':  1}}
r = risk_score(ERRORS, COSTS)
assert r['停车让行'] == 2050 and r['限速60'] == 3600 and r['景点指示'] == 350
rank = sorted(r, key=lambda k: -r[k])
assert rank == ['限速60', '停车让行', '景点指示']
assert r['停车让行'] > r['景点指示'], '错误数少 100 倍，风险却高 6 倍'

n_err = {k: v['fn'] + v['fp'] for k, v in ERRORS.items()}
print(f"{'类别':<10s} {'错误总数':>9s} {'风险分':>9s}  {'按错误数排名':>12s} {'按风险排名':>10s}")
by_n = sorted(n_err, key=lambda k: -n_err[k])
for k in ERRORS:
    print(f'{k:<10s} {n_err[k]:>9d} {r[k]:>9d} {by_n.index(k)+1:>12d} {rank.index(k)+1:>10d}')
print('\n✅ 练习 3 通过。两个必须记住的结论：')
print('   ① **按错误数排序与按风险排序的结果完全相反**（景点指示错最多，风险最低）。')
print('      按「哪个类 badcase 最多」安排工作，就是在给最不重要的类投入最多人力。')
print('   ② 风险 = 频率 x 代价，两项都要算。「限速60」之所以排第一，')
print('      不是因为它最危险，而是因为它**又常见又不便宜** —— 这才是真实的优先级。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def required_hfov_deg(size_m, dist_m, min_px, img_w=1920):
    f = min_px * dist_m / size_m                       # 所需等效焦距（像素）
    return 2.0 * math.degrees(math.atan((img_w / 2.0) / f))

In [ ]:
# 练习 2 参考答案
def approach_frames(z_far, z_near, speed_kmh, fps=30.0):
    t = (z_far - z_near) / (speed_kmh / 3.6)           # 接近过程时长（秒）
    return math.floor(t * fps + 1e-9)

In [ ]:
# 练习 3 参考答案
def risk_score(errors, costs):
    return {c: errors[c]['fn'] * costs[c]['fn'] + errors[c]['fp'] * costs[c]['fp']
            for c in errors}

---
## 🧪 真实工程胶囊：一页纸的 TSR 感知规格书

下面这份 `RECIPE` 可以原样复制到真实项目里作为**感知需求文档的骨架**。
它把本模块的四件事（软件栈位置 / 五个本质不同 / 四层输出 / 像素预算）
落成可评审、可验收的条目。

In [ ]:
RECIPE = r'''
# ============================================================
# TSR 感知规格书（骨架）  v0.1
# ============================================================

## 1. 硬件与像素预算  ——  先算物理，再谈模型
camera:
  main:  {resolution: [1920, 1080], hfov_deg: 60,  f_px: 1663}
  tele:  {resolution: [1920, 1080], hfov_deg: 30,  f_px: 3582}
  wide:  {resolution: [1920, 1080], hfov_deg: 120, f_px: 554}

sign_physical_size_m:      # 依据 GB 5768
  prohibitory_circle_urban: 0.60
  prohibitory_circle_road:  0.80
  warning_triangle:         0.70

# 像素预算（60cm 标志）：p = f_px * S / Z
#   主摄: 50m->20px  60m->17px  100m->10px  146m->6.8px
#   长焦: 50m->43px 100m->21px  146m->15px
# 结论：>100m 的 L2 细类识别必须依赖长焦；主摄在 >60m 只承诺 L1 粗类。

## 2. 需求反推  ——  安全需要多远
required_detection_distance_m:
  # D = v0*t_r + (v0^2 - v1^2) / (2a)，t_r=0.8s，a=2.0 m/s^2
  highway_100_to_60: 146
  highway_120_to_100: 106
  urban_60_to_30:     70
# ⚠️ 146m 处主摄仅 6.8px —— 单帧不可解。必须：长焦 + 时序累积 + 地图先验。

## 3. 输出 schema  ——  四层，缺一层下游就用不了
TsrDetection:
  # L0 定位
  bbox_xyxy:        [float, float, float, float]
  camera_id:        str            # 多相机融合必须
  timestamp_ns:     int            # 时序关联必须
  det_confidence:   float          # ⚠️ 与 cls_confidence **分开输出，不要相乘**
  # L1 粗类（可降级：细类不确定时仍然输出这一层）
  coarse_class:     enum[prohibitory, warning, mandatory, guide, auxiliary, workzone, vms]
  coarse_confidence: float
  # L2 细类
  fine_class:       enum[...] | UNKNOWN
  cls_confidence:   float
  # L3 属性
  attributes:
    speed_value:      int   | null      # 限速值
    time_range:       str   | null      # "08:00-18:00"
    vehicle_type:     str   | null      # "truck"
    arrow_direction:  str   | null
    condition:        enum[normal, faded, damaged, occluded, graffiti]
    facing:           enum[front, back, side]      # ← 对向车道判定的几何证据
    is_variable:      bool                         # 可变电子牌
    group_id:         int   | null      # 组合牌（主牌 + 辅助牌）的关联
  # 时序（由融合层填充，感知层留空）
  track_id:         int | null
  first_seen_dist_m: float | null
  accumulated_confidence: float | null

## 4. 验收指标  ——  mAP 只是入场券
acceptance:
  bucketed_recall:                  # 按像素尺寸分桶，否则改进不可见
    ">= 32px": 0.98
    "16-32px": 0.92
    "8-16px":  0.70                 # L1 粗类召回；L2 不做承诺
  critical_class_recall:            # 关键类单独立指标
    stop_giveway: 0.995
  false_positive_per_km: "< 0.05"   # ← 比 precision 有用得多
  first_detection_distance_p50_m:
    speed_limit: 90
  flicker_per_instance: "< 0.5"     # 上报后又撤销的次数
  latency_p99_ms: 25                # 尾延迟，不是均值

## 5. 明确不做的事（写清楚边界，避免下游误用）
non_goals:
  - 感知层不判断标志是否对自车生效；只输出 facing / 横向位置 / 3D 位置等几何证据
  - 感知层不做限速值的合法性校验（如 "限速 999"）；由下游规则层兜底
  - 可变电子牌只输出「这是 VMS + 当前读数」，不承诺读数的时效性
'''
print(RECIPE)
for k in ['f_px', 'GB 5768', 'required_detection_distance_m', 'det_confidence',
          'coarse_class', 'facing', 'false_positive_per_km', 'non_goals']:
    assert k in RECIPE, k
print('✅ 规格书覆盖：像素预算 / 需求反推 / 四层输出 schema / 分桶验收 / 边界声明')

### 小结

- **先算物理，再谈模型。** 针孔模型 `p = f_px·S/Z` 与制动公式 `D = v₀t_r + (v₀²-v₁²)/2a`
  这两行，就能定出一套 TSR 方案的上界。**面试里先算这两个数，再谈方案。**
- **本课的中心矛盾**：安全需要 146 m 的检出距离，主摄在那里只给 6.8 px。
  出路只有 **长焦 / 时序 / 地图** 三条，**没有一条是「换更大的 backbone」**。
- **五个本质不同：小 · 偏 · 硬 · 连 · 歪。** 目标小（16 px 是常态）、分布偏（且尾巴随地域搬家）、
  语义硬（直接变控制约束）、时序连（静止刚体 + 自车运动已知）、代价歪（双重不对称）。
  **前三点是问题难度，第五点是损失函数形状，第四点是额外先验——用第四点去补前三点。**
- **TSR 的输出是四层**（定位 / 粗类 / 细类 / 属性），不是「框 + 类别」。
  L1 存在的理由是**可降级**；L3 里的 `facing` 与车道关联是最容易漏、也最容易加分的两项。
  **两个置信度要分开输出，不要相乘。**
- **感知层如实输出所有标志与几何证据，「这块牌是否对自车生效」由有上下文的下游判断。**
  把这个判断塞进感知层是常见的架构错误。
- **按错误数排优先级与按风险排优先级的结论常常相反。** 风险 = 频率 × 代价，两项都要算。

下一站：**模块 01 · 数据集与标志分类体系** —— 你的类别体系决定了你的天花板。